## Import Libraries

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

## Load Environment Variables

In [2]:
load_dotenv(override=True)

True

## Add Pydantic Model for Evaluation

In [3]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

## System Prompt and Evaluator Prompts

In [4]:
system_prompt = f"You are acting as GrocerAI, an AI assistant for a grocery store. You are answering questions from customers on the store's website"

In [5]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality and friendly."

evaluator_system_prompt += "\n\n## Here's the Conversation:\n"

## OpenAI client

In [6]:
openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Evaluator

In [7]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [8]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.chat.completions.parse(model="gpt-4.1-nano", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

## Rerun

In [9]:
def rerun(reply, message, history, feedback):

    print("\nOriginal reply:", reply)
    print("Original message:", message)
    print ("history:", history)
    print("Rerunning with feedback:",)
    
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"

    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    
    return response.choices[0].message.content

In [10]:
def chat(message, history):

    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply = response.choices[0].message.content
    
    # override for testing
    reply = "Apples are healthy!" 

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [11]:
chat("Is Milkshakes healthy?", [])

Failed evaluation - retrying
The response is not relevant to the user's question about the healthiness of milkshakes. It provides information about apples instead, which may be confusing or unhelpful.

Original reply: Apples are healthy!
Original message: Is Milkshakes healthy?
history: []
Rerunning with feedback:


"Milkshakes can vary in healthiness depending on their ingredients. Traditional milkshakes made with ice cream, milk, and sweeteners can be high in sugar and calories. However, if made with healthier ingredients like low-fat milk, yogurt, or fruits, they can offer some nutritional benefits. It's best to enjoy milkshakes in moderation and consider the ingredients to make a healthier choice."